# Deliverable D5: Semantic Product Retrieval (MiniLM + FAISS)

This notebook builds and evaluates the semantic product retrieval subsystem for SmartCart:
1. **Frozen Semantic Encoder**: `sentence-transformers/all-MiniLM-L6-v2` with L2-normalized product embeddings.
2. **Exact Inner-Product Index**: `faiss.IndexFlatIP` for cosine ranking on normalized vectors.
3. **Post-Retrieval Eligibility Filtering**: Strict ingredient identity, unit, and explicit dietary suitability checks.
4. **150-Query Benchmark**: Evaluates TF-IDF Cosine vs MiniLM + FAISS on 50 development and 100 test queries (synonyms, brands, misspellings, near-matches) using Recall@5, Recall@10, and MRR.


In [ ]:
# Environment setup and modular retrieval imports
os=__import__('os')
sys=__import__('sys')
json=__import__('json')
Path=__import__('pathlib').Path
plt=__import__('matplotlib.pyplot',fromlist=['pyplot'])
os.environ['KMP_DUPLICATE_LIB_OK']='TRUE'
sys.path.insert(0,str(Path.cwd()/'src'))
from	smartcart.data.contracts	import	load_pilot
from	smartcart.retrieval.tfidf	import	build_tfidf_retriever,search_products_tfidf
from	smartcart.retrieval.semantic	import	load_semantic_encoder,build_faiss_index,search_products_semantic
from	smartcart.retrieval.filtering	import	filter_eligible_candidates
from	smartcart.retrieval.runner	import	execute_retrieval_benchmark
print('Modular	retrieval	subsystem	loaded')


### 1. Load Pilot Catalog & Build TF-IDF + FAISS Indices


In [ ]:
# Build TF-IDF baseline and MiniLM FAISS semantic index
catalog=load_pilot(Path('data/processed/pilot/catalog.json'))
products=catalog['products']
tfidf_state=build_tfidf_retriever(products)
encoder=load_semantic_encoder()
faiss_state=build_faiss_index(products,encoder)
print('Indexed	products:',len(products))


### 2. Per-Ingredient Semantic Search & Strict Dietary Filtering


In [ ]:
# Query synonym and verify strict dietary eligibility filtering
cands=search_products_semantic('poultry	fillet',faiss_state,encoder,top_k=10)
print('Top	semantic	match:',cands[0]['id'],round(cands[0]['retrieval_score'],4))
veg_only=filter_eligible_candidates(cands,dietary_rules=['vegetarian'])
print('Eligible	after	vegetarian	filter:',len(veg_only))


### 3. 150-Query Benchmark (50 Dev / 100 Test): TF-IDF vs MiniLM


In [ ]:
# Run retrieval benchmark and print Recall@5, Recall@10, and MRR
results=execute_retrieval_benchmark(output_path='results/predictions/retrieval_benchmark_results.json')
print('Dev	TF-IDF:',results['dev']['tfidf'])
print('Dev	MiniLM:',results['dev']['minilm'])
print('Test	TF-IDF:',results['test']['tfidf'])
print('Test	MiniLM:',results['test']['minilm'])


### 4. Retrieval Performance Comparison Visualization


In [ ]:
# Render retrieval benchmark figure
img=plt.imread('results/figures/11_retrieval_tfidf_vs_minilm.png')
fig=plt.figure(figsize=(10,5),dpi=150)
plt.imshow(img)
plt.axis('off')
plt.show()
